In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train = DataLoader(datasets.FashionMNIST('.', train=True,  download=True, transform=transform), batch_size=64, shuffle=True)
test  = DataLoader(datasets.FashionMNIST('.', train=False, download=True, transform=transform), batch_size=256)

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
    nn.Linear(128, 10)
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

In [ ]:
optimizer = optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    for x, y in train:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss_fn(model(x), y).backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        correct = sum((model(x.to(device)).argmax(1) == y.to(device)).sum().item() for x, y in test)
    print(f"Epoch {epoch+1}  acc={correct/len(test.dataset):.3f}")